# Özdeğer ve Özvektör Hesaplama

Bu notebook'ta özdeğer (eigenvalue) hesaplamasını iki farklı yöntemle yapacağız:
1. **Manuel yöntem**: NumPy'ın `eig` fonksiyonunu kullanmadan, karakteristik denklem üzerinden hesaplama
2. **NumPy yöntemi**: `numpy.linalg.eig` fonksiyonu ile hesaplama

Sonuçları karşılaştıracağız.

**Referans:** [LucasBN/Eigenvalues-and-Eigenvectors](https://github.com/LucasBN/Eigenvalues-and-Eigenvectors)

In [1]:
import numpy as np

## 1. Manuel Özdeğer Hesaplama Fonksiyonları

Aşağıdaki fonksiyonlar LucasBN'nin reposundan referans alınarak yazılmıştır. Temel mantık şu şekilde çalışır:

- Matrisin boyutlarını bul
- Birim matrisi (I) oluştur
- Karakteristik denklemi kur: `det(A - λI) = 0`
- Determinantı polinom olarak hesapla
- Polinomun köklerini bul → bu kökler özdeğerlerdir

In [2]:
def get_dimensions(matrix):
    """Matrisin satır ve sütun sayısını döndürür."""
    return [len(matrix), len(matrix[0])]


def identity_matrix(dimensions):
    """Verilen boyutlarda birim matris oluşturur."""
    matrix = [[0 for j in range(dimensions[1])] for i in range(dimensions[0])]
    for i in range(dimensions[0]):
        matrix[i][i] = 1
    return matrix


def list_multiply(list1, list2):
    """
    İki polinomu çarpar.
    Örnek: [2, 1] * [3, 1] = [6, 5, 1] yani (2+x)*(3+x) = 6 + 5x + x^2
    """
    result = [0 for _ in range(len(list1) + len(list2) - 1)]
    for i in range(len(list1)):
        for j in range(len(list2)):
            result[i + j] += list1[i] * list2[j]
    return result


def list_add(list1, list2, sub=1):
    """İki polinomu toplar veya çıkarır (sub=-1 ise çıkarma)."""
    return [i + (sub * j) for i, j in zip(list1, list2)]


def find_determinant(matrix, excluded=1):
    """Kofaktör açılımı ile determinant hesaplar (sayısal)."""
    dimensions = get_dimensions(matrix)
    if dimensions == [2, 2]:
        return excluded * ((matrix[0][0] * matrix[1][1]) - (matrix[0][1] * matrix[1][0]))
    else:
        new_matrices = []
        excluded_vals = []
        exclude_row = 0
        for exclude_column in range(dimensions[1]):
            tmp = []
            excluded_vals.append(matrix[exclude_row][exclude_column])
            for row in range(1, dimensions[0]):
                tmp_row = []
                for column in range(dimensions[1]):
                    if (row != exclude_row) and (column != exclude_column):
                        tmp_row.append(matrix[row][column])
                tmp.append(tmp_row)
            new_matrices.append(tmp)
        determinants = [find_determinant(new_matrices[j], excluded_vals[j]) for j in range(len(new_matrices))]
        determinant = 0
        for i in range(len(determinants)):
            determinant += ((-1) ** i) * determinants[i]
        return determinant

In [3]:
def characteristic_equation(matrix):
    """
    Karakteristik denklemi oluşturur: A - λI
    Her eleman [a, -b] formatında bir polinom olur.
    Örneğin köşegen elemanı 6 ise [6, -1] olur (6 - λ).
    """
    dimensions = get_dimensions(matrix)
    I = identity_matrix(dimensions)
    return [[[a, -b] for a, b in zip(row_a, row_i)] for row_a, row_i in zip(matrix, I)]


def determinant_equation(matrix, excluded=[1, 0]):
    """
    Determinantı polinom olarak hesaplar.
    Sonuç: [c0, c1, c2, ...] -> c0 + c1*λ + c2*λ^2 + ...
    """
    dimensions = get_dimensions(matrix)
    if dimensions == [2, 2]:
        tmp = list_add(
            list_multiply(matrix[0][0], matrix[1][1]),
            list_multiply(matrix[0][1], matrix[1][0]),
            sub=-1
        )
        return list_multiply(tmp, excluded)
    else:
        new_matrices = []
        excluded_vals = []
        exclude_row = 0
        for exclude_column in range(dimensions[1]):
            tmp = []
            excluded_vals.append(matrix[exclude_row][exclude_column])
            for row in range(1, dimensions[0]):
                tmp_row = []
                for column in range(dimensions[1]):
                    if (row != exclude_row) and (column != exclude_column):
                        tmp_row.append(matrix[row][column])
                tmp.append(tmp_row)
            new_matrices.append(tmp)
        determinant_equations = [determinant_equation(new_matrices[j], excluded_vals[j]) 
                                 for j in range(len(new_matrices))]
        dt_equation = [sum(i) for i in zip(*determinant_equations)]
        return dt_equation


def find_eigenvalues(matrix):
    """
    Özdeğerleri hesaplar.
    Karakteristik polinomu oluşturup np.roots ile köklerini bulur.
    """
    dt_equation = determinant_equation(characteristic_equation(matrix))
    # np.roots en yüksek dereceden başlayan katsayı bekler, o yüzden ters çeviriyoruz
    return np.roots(dt_equation[::-1])

## 2. Test Matrisi Tanımlama

Referans repodaki aynı 3x3 matrisi kullanacağız:

In [4]:
# Test matrisi (referans repodan)
A = [[6, 1, -1],
     [0, 7, 0],
     [3, -1, 2]]

print("Matris A:")
for row in A:
    print(row)

Matris A:
[6, 1, -1]
[0, 7, 0]
[3, -1, 2]


## 3. Manuel Yöntem ile Özdeğer Hesaplama

In [5]:
# Manuel yöntemle özdeğer hesaplama
manuel_eigenvalues = find_eigenvalues(A)
print("Manuel Yöntem ile Özdeğerler:")
print(manuel_eigenvalues)
print()
# Her özdeğeri tek tek gösterelim
for i, val in enumerate(manuel_eigenvalues):
    print(f"  λ{i+1} = {val.real:.4f}")

Manuel Yöntem ile Özdeğerler:
[7. 5. 3.]

  λ1 = 7.0000
  λ2 = 5.0000
  λ3 = 3.0000


## 4. NumPy eig Fonksiyonu ile Özdeğer ve Özvektör Hesaplama

In [6]:
# NumPy ile özdeğer ve özvektör hesaplama
A_np = np.array(A)
numpy_eigenvalues, numpy_eigenvectors = np.linalg.eig(A_np)

print("NumPy eig ile Özdeğerler:")
print(numpy_eigenvalues)
print()
for i, val in enumerate(numpy_eigenvalues):
    print(f"  λ{i+1} = {val:.4f}")

print("\nNumPy eig ile Özvektörler:")
print(numpy_eigenvectors)
print()
for i in range(len(numpy_eigenvalues)):
    print(f"  λ{i+1} = {numpy_eigenvalues[i]:.4f} için özvektör: {numpy_eigenvectors[:, i]}")

NumPy eig ile Özdeğerler:
[5. 3. 7.]

  λ1 = 5.0000
  λ2 = 3.0000
  λ3 = 7.0000

NumPy eig ile Özvektörler:
[[0.70710678 0.31622777 0.58834841]
 [0.         0.         0.78446454]
 [0.70710678 0.9486833  0.19611614]]

  λ1 = 5.0000 için özvektör: [0.70710678 0.         0.70710678]
  λ2 = 3.0000 için özvektör: [0.31622777 0.         0.9486833 ]
  λ3 = 7.0000 için özvektör: [0.58834841 0.78446454 0.19611614]


## 5. Sonuçların Karşılaştırılması

In [7]:
# Sonuçları sıralayıp karşılaştıralım
manuel_sorted = sorted(manuel_eigenvalues.real)
numpy_sorted = sorted(numpy_eigenvalues.real)

print("=" * 50)
print("SONUÇ KARŞILAŞTIRMASI")
print("=" * 50)
print(f"{'Özdeğer':<10} {'Manuel':<15} {'NumPy eig':<15} {'Fark':<15}")
print("-" * 50)
for i in range(len(manuel_sorted)):
    fark = abs(manuel_sorted[i] - numpy_sorted[i])
    print(f"λ{i+1:<9} {manuel_sorted[i]:<15.6f} {numpy_sorted[i]:<15.6f} {fark:<15.2e}")

print("-" * 50)
print("\nİki yöntem de aynı özdeğerleri buluyor.")
print("Manuel yöntem sadece özdeğerleri hesaplarken,")
print("NumPy'ın eig fonksiyonu hem özdeğerleri hem özvektörleri döndürür.")

SONUÇ KARŞILAŞTIRMASI
Özdeğer    Manuel          NumPy eig       Fark           
--------------------------------------------------
λ1         3.000000        3.000000        6.22e-15       
λ2         5.000000        5.000000        2.40e-14       
λ3         7.000000        7.000000        2.04e-14       
--------------------------------------------------

İki yöntem de aynı özdeğerleri buluyor.
Manuel yöntem sadece özdeğerleri hesaplarken,
NumPy'ın eig fonksiyonu hem özdeğerleri hem özvektörleri döndürür.


## 6. Doğrulama: A * v = λ * v

Özdeğer ve özvektörlerin doğruluğunu kontrol edelim. Eğer `A * v = λ * v` eşitliği sağlanıyorsa sonuçlar doğrudur.

In [8]:
# Doğrulama: A * v = λ * v olmalı
print("Doğrulama (A*v = λ*v):")
print("-" * 40)
for i in range(len(numpy_eigenvalues)):
    v = numpy_eigenvectors[:, i]
    lam = numpy_eigenvalues[i]
    
    sol_taraf = A_np @ v       # A * v
    sag_taraf = lam * v        # λ * v
    
    print(f"\nλ{i+1} = {lam:.4f}")
    print(f"  A * v  = {sol_taraf}")
    print(f"  λ * v  = {sag_taraf}")
    print(f"  Eşit mi? {np.allclose(sol_taraf, sag_taraf)}")

Doğrulama (A*v = λ*v):
----------------------------------------

λ1 = 5.0000
  A * v  = [3.53553391 0.         3.53553391]
  λ * v  = [3.53553391 0.         3.53553391]
  Eşit mi? True

λ2 = 3.0000
  A * v  = [0.9486833  0.         2.84604989]
  λ * v  = [0.9486833  0.         2.84604989]
  Eşit mi? True

λ3 = 7.0000
  A * v  = [4.11843884 5.49125178 1.37281295]
  λ * v  = [4.11843884 5.49125178 1.37281295]
  Eşit mi? True
